# BATDiff method check on a DIV2K high-resolution photograph

I use this notebook to ask whether the public BATDiff code can copy a sharp natural photo.

This is not super-resolution. I give BATDiff a 256x256 high-resolution crop from `data/DIV2K_valid_HR/` and set `sr_factor = 1`. If the method works, the output should still look like that photo, not like noise.

| Setting | Value |
|---|---|
| Input | one photo from `data/DIV2K_valid_HR/`, centre-cropped to 256 |
| Task | copy the crop (`sr_factor = 1`) |
| GPU | Colab T4 |

BATDiff trains on one image at a time. I do not upload the whole 100-image folder. A full 2040x1356 photo with `sr_factor=4` would try to make an ~8160x5424 image and run out of memory.

I set the runtime to T4 GPU. A ready 256 crop is `outputs/sanity/batdiff_hr_identity/0806_crop256.png`. I first leave `SMOKE_TEST = True` (a few minutes; the picture may look like noise). If that cell finishes without an error, I set it to `False`, restart the runtime, and run from the top again.

The long T4 check continues from 20000 to 60000 steps (~8 hours). I upload the palm crop and `model-8.pt` in Step 2. I leave the Colab tab open until `training completed`, when the notebook downloads a zip.


In [ ]:
#@title Step 0 — check the runtime has a GPU
import torch

if not torch.cuda.is_available():
    raise SystemExit(
        "No GPU visible.\n"
        "Fix: Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU, "
        "then run this cell again."
    )

total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU:   {torch.cuda.get_device_name(0)}")
print(f"VRAM:  {total_gb:.1f} GB")
print(f"torch: {torch.__version__}")

In [ ]:
#@title Step 1 — clone BATDiff and install four packages
import os, subprocess
from pathlib import Path

PROJECT_REPO = "https://github.com/yoyowuyogwrt-hue/3D-OCT-Image-SuperResolution-Benchmark"
BATDIFF_REPO = "https://github.com/MaryamHeidari-1994/BATDiff"

WORK = Path("/content")
BATDIFF = WORK / "BATDiff"
PROJECT = WORK / "project"


def sh(cmd, cwd=None):
    print(f"$ {cmd}")
    result = subprocess.run(
        cmd, shell=True, cwd=cwd, text=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    )
    print(result.stdout)
    if result.returncode:
        raise RuntimeError(f"command failed with exit code {result.returncode}")


if not BATDIFF.exists():
    sh(f"git clone --depth 1 {BATDIFF_REPO} {BATDIFF}")
if not PROJECT.exists():
    sh(f"git clone --depth 1 {PROJECT_REPO} {PROJECT}")

sh("pip install -q einops ftfy regex PyWavelets lpips")
sh(f"python {PROJECT}/scripts/batdiff_dip_patch.py --batdiff-root {BATDIFF}")
print("Patch applied for Colab compatibility. This run omits --xref_image.")
print("BATDiff:", BATDIFF)

## Step 2 — Upload the palm crop and the 20k checkpoint

When the cell asks, I choose both files:

- `outputs/sanity/batdiff_hr_identity/0806_crop256.png`
- `outputs/sanity/batdiff_hr_identity/overnight_20k/hr_identity/model-8.pt`

If I still have the unzip on this Mac, `model-8.pt` is also inside `Downloads/hr_identity 3/hr_identity/`.


In [ ]:
#@title Step 2 — upload the palm crop and the 20k checkpoint
from google.colab import files
from PIL import Image
from IPython.display import display
import io

CROP_SIZE = 256
DATA = WORK / "data" / "hr_identity"
DATA.mkdir(parents=True, exist_ok=True)
UPLOADED_CKPT = None

print("Upload BOTH: 0806_crop256.png and model-8.pt")
uploaded = files.upload()
if len(uploaded) < 1:
    raise FileNotFoundError("Upload 0806_crop256.png and model-8.pt")

image = None
for raw_name, raw_bytes in uploaded.items():
    lower = raw_name.lower()
    if lower.endswith(".pt") or "model-8" in lower:
        UPLOADED_CKPT = DATA / "model-8.pt"
        UPLOADED_CKPT.write_bytes(raw_bytes)
        print(f"checkpoint: {raw_name} -> {UPLOADED_CKPT} ({len(raw_bytes)/1e6:.1f} MB)")
    elif lower.endswith((".png", ".jpg", ".jpeg")):
        image = Image.open(io.BytesIO(raw_bytes)).convert("RGB")
        print(f"image: {raw_name}: {image.size[0]}x{image.size[1]}")

if image is None:
    raise FileNotFoundError("Missing the palm PNG. Upload 0806_crop256.png as well as model-8.pt.")

width, height = image.size
if width < CROP_SIZE or height < CROP_SIZE:
    raise ValueError(f"Image {image.size} is smaller than the {CROP_SIZE} crop.")
left = (width - CROP_SIZE) // 2
top = (height - CROP_SIZE) // 2
crop = image.crop((left, top, left + CROP_SIZE, top + CROP_SIZE))
HR = DATA / "hr.png"
crop.save(HR)
print(f"saved identity input {CROP_SIZE}x{CROP_SIZE} -> {HR}")
if UPLOADED_CKPT is None:
    print("WARNING: no model-8.pt. Training will start from step 0, not from 20k.")
display(crop)

In [ ]:
#@title Step 3 — BATDiff identity settings
SMOKE_TEST = False  #@param {type:"boolean"}
SAVE_TO_DRIVE = False  #@param {type:"boolean"}
CONTINUE_FROM_20K = True  #@param {type:"boolean"}

SR_FACTOR = 1  # identity: do not enlarge the HR crop
if SMOKE_TEST:
    DIM, TRAIN_STEPS, TIMESTEPS, TS, SAVE_EVERY = 200, 101, 100, 4, 101
    LOAD_MILESTONE = 0
elif CONTINUE_FROM_20K:
    # Resume model-8 (step 20000) and train until 60000 (~8 h more on T4).
    DIM, TRAIN_STEPS, TIMESTEPS, TS, SAVE_EVERY = 200, 60000, 100, 4, 2500
    LOAD_MILESTONE = 8
else:
    DIM, TRAIN_STEPS, TIMESTEPS, TS, SAVE_EVERY = 200, 20000, 100, 4, 2500
    LOAD_MILESTONE = 0

if SAVE_TO_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        RESULTS = Path("/content/drive/MyDrive/batdiff_hr_identity")
    except Exception as error:
        print("Drive mount failed:", error)
        print("Saving inside this Colab session instead.")
        RESULTS = WORK / "results_hr_identity"
else:
    RESULTS = WORK / "results_hr_identity"
RESULTS.mkdir(parents=True, exist_ok=True)

COMMON_FLAGS = (
    f"--mode train "
    f"--image_name hr.png "
    f"--use_atrous --atrous_wavelet b3 --atrous_level 6 "
    f"--sr_factor {SR_FACTOR} "
    f"--dim {DIM} "
    f"--ts {TS} "
    f"--train_num_steps {TRAIN_STEPS} "
    f"--timesteps {TIMESTEPS} "
    f"--save_and_sample_every {SAVE_EVERY} "
)
if LOAD_MILESTONE:
    COMMON_FLAGS += f"--load_milestone {LOAD_MILESTONE} "

print("SMOKE TEST" if SMOKE_TEST else "CONTINUE 20k → 60k" if CONTINUE_FROM_20K else "IDENTITY RUN")
print(f"  dim         {DIM}")
print(f"  ts          {TS}")
print(f"  train steps {TRAIN_STEPS}")
print(f"  load        {LOAD_MILESTONE}")
print(f"  save every  {SAVE_EVERY}")
print(f"  sr_factor   {SR_FACTOR}")
print(f"  results     {RESULTS}")
if CONTINUE_FROM_20K:
    print("\nKeep this Colab tab open. Do not close the Mac lid.")
    print("Expect step:20100, step:20200, ... until step:59900, then 'training completed'.")

In [ ]:
#@title Step 4 — run BATDiff identity and download the zip
import re, sys, time, shutil
from google.colab import files


def find_final_image(scope_dir: Path) -> Path:
    candidates = list((scope_dir / "final_samples").glob("*.png"))
    if not candidates:
        raise FileNotFoundError(f"No samples under {scope_dir / 'final_samples'}")

    def scale_of(path: Path) -> int:
        match = re.search(r"_s(\d+)_", path.name)
        return int(match.group(1)) if match else -1

    finest = max(scale_of(p) for p in candidates)
    at_finest = [p for p in candidates if scale_of(p) == finest]
    return max(at_finest, key=lambda p: p.stat().st_mtime)


tag = "hr_identity"
scope_dir = RESULTS / tag / tag
scope_dir.mkdir(parents=True, exist_ok=True)
if LOAD_MILESTONE and "UPLOADED_CKPT" in dir() and UPLOADED_CKPT is not None:
    dest = scope_dir / f"model-{LOAD_MILESTONE}.pt"
    shutil.copy(UPLOADED_CKPT, dest)
    print("loaded checkpoint into", dest)
elif LOAD_MILESTONE:
    raise FileNotFoundError("CONTINUE_FROM_20K is on, but model-8.pt was not uploaded in Step 2.")

flags = COMMON_FLAGS + (
    f"--scope {tag} --dataset_folder {DATA}/ --results_folder {RESULTS / tag} "
)
print("flags:", flags)

started = time.time()
process = subprocess.Popen(
    f"PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python main.py {flags}",
    shell=True, cwd=BATDIFF, text=True,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1,
)
for line in process.stdout:
    sys.stdout.write(line)
process.wait()
if process.returncode:
    raise RuntimeError(f"BATDiff failed with exit code {process.returncode}")

out = find_final_image(scope_dir)
print(f"\nfinished in {(time.time() - started) / 60:.1f} min -> {out.name}")

archive = shutil.make_archive(str(WORK / "batdiff_hr_identity"), "zip", RESULTS)
print("zip:", archive)
try:
    files.download(archive)
except Exception as error:
    print(f"(automatic download unavailable: {error}; use the Files pane on the left)")

In [ ]:
#@title Step 5 — compare the HR crop with BATDiff (pass = looks like a photo)
import sys
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
from src.metrics.evaluate import evaluate

hr = np.asarray(Image.open(HR).convert("RGB"))
bat = np.asarray(Image.open(out).convert("RGB"))
if bat.shape != hr.shape:
    bat = np.asarray(Image.open(out).convert("RGB").resize((hr.shape[1], hr.shape[0]), Image.BICUBIC))

scores = evaluate(hr, bat)
print(f"BATDiff vs HR crop: PSNR={scores['PSNR']:.4f}  SSIM={scores['SSIM']:.4f}  LPIPS={scores['LPIPS']:.4f}")
print("Pass/fail is visual: does the right panel look like the left photo, or like noise?")

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(hr)
axes[0].set_title("HR crop (input)")
axes[0].axis("off")
axes[1].imshow(bat)
axes[1].set_title("BATDiff identity")
axes[1].axis("off")
axes[1].text(
    0.5, -0.04,
    f"PSNR {scores['PSNR']:.2f}  SSIM {scores['SSIM']:.3f}  LPIPS {scores['LPIPS']:.3f}",
    transform=axes[1].transAxes, ha="center", va="top", fontsize=9,
)
fig.tight_layout()
figure_path = RESULTS / ("comparison_smoke.png" if SMOKE_TEST else "comparison.png")
fig.savefig(figure_path, dpi=150, bbox_inches="tight")
plt.show()
print("saved", figure_path)

if SMOKE_TEST:
    print("Smoke-test output is not a result. Set SMOKE_TEST=False, restart, and run all cells again.")

## If something fails

**No GPU** — I set Runtime → Change runtime type → T4 GPU.

**CUDA out of memory** — I keep the 256 crop. I do not upload a full 2K photo or raise `CROP_SIZE`. If it still fails, I set `DIM` to 128 and run Step 3 then Step 4 again.

**Smoke test looks like noise** — expected. Only `SMOKE_TEST = False` counts.

**Session died overnight** — I leave the Colab tab open, with the Mac plugged in and sleep off. If I ticked `SAVE_TO_DRIVE` in Step 3, checkpoints are in Google Drive even if the VM dies. I look for `step:100`, `step:200`, and so on.

**Smoke-test output is not a result** — I set `SMOKE_TEST=False`, restart, and run all cells again.

After a successful full run I copy the zip into `outputs/sanity/batdiff_hr_identity/` on my Mac. If I used Drive, the folder is `Google Drive / batdiff_hr_identity`.
